# Workflow for PDMS Molecular Dynamics Simulations Using LAMMPS

This notebook demonstrates a workflow for building, simulating, and analysing **polydimethylsiloxane (PDMS)** systems using LAMMPS.

### 1. Polymer Construction

- PDMS chains are generated using **SwiftPol**:  
  https://github.com/matta-research-group/SwiftPol

- The generated chains are converted into:
  - OpenFF `Molecule` --> `Topology` --> `Interchange` objects

- Force field parameters are assigned using a custom `bespoke_ff.offxml` force field generated with **Presto**:  
  https://github.com/cole-group/presto

### 2. Initial Configuration Generation

- **Polyply** is used to construct the initial simulation box and pack the polymer chains into a system:  
  https://github.com/marrink-lab/polyply_1.0

### 3. Molecular Dynamics Simulations

LAMMPS is used as the simulation engine (although the workflow could be adapted to use **GROMACS** or **OpenMM**).

The simulation protocol consists of:

1. **Energy minimisation**
2. **Heating** to **600 K**
3. **Cooling** to **300 K**
4. **NPT MD** at **300 K** **10 ns**

Heating above the PDMS glass transition temperature promotes equilibration before slow cooling.

### 4. Output Files

`.lammpstrj` trajectory file containing xyz coordinates \
`.thermo` thermo dynamic outputs \
`stats.dat` volume and enthalpy data from every 100 steps 

This data can be analysed directly and visualised with tools such as **ASE**, **MDAnalysis**, **NumPy**, **Ovito** and **Matplotlib** .

### 5. Analysis

Post-processing includes:

- Density plot
- Volume fluctuation analysis
- Compressability / Bulk Modulus  
- Specific Heat Capacity
- Viscosity

In [ ]:
from swiftpol import build
from openff.toolkit import Molecule, Topology, unit
from openff.interchange import Interchange
from openff.toolkit.typing.engines.smirnoff import ForceField
import random
from rdkit import Chem
import subprocess
import numpy as np

In [ ]:
# Swiftpol used to build a system of 10 PDMS chains of 40 monomers.
sys = build.polymer_system_from_PDI(
            monomer_list=['I-[Si](-C)(-C)-O-I'],
            reaction='[O:1][I:3].[Si:2][I:4]>>[O:1][Si:2].[I:3][I:4]',
            length_target=40,
            terminals='C',
            num_chains=10,
            PDI_target=1.0,
            acceptance=5
        )

# Small fix to replace -OH terminal group with SiMe3 group (-OH terminals not yet paramaterized in the bespoke force field) 
def terminal_fix(sys):
    # rdmol_chain = sys.chain_rdkit[0]
    for x in range(len(sys.chain_rdkit)):
        rdmol_chain = sys.chain_rdkit[x]
        # initialize new terminal group
        terminal = Chem.MolFromSmiles("[Si](-C)(-C)(-C)")
        terminal = Chem.AddHs(terminal)
        rdmol_chain = Chem.RemoveAllHs(rdmol_chain)
        #Copy over old atom information to new terminal group (important if you're using polyply down the line)
        index_old_atom = rdmol_chain.GetSubstructMatch(Chem.MolFromSmarts("[Si](-[OH])(-C)(-C)"))[0]
        old_atom = rdmol_chain.GetAtomWithIdx(index_old_atom)
        info = old_atom.GetPDBResidueInfo()
        [atom.SetMonomerInfo(info) for atom in terminal.GetAtoms()]
        # replace old terminal group with new one
        rdmol_chain = Chem.ReplaceSubstructs(rdmol_chain, Chem.MolFromSmarts("[Si](-[OH])(-C)(-C)"), terminal)[0]
        Chem.SanitizeMol(rdmol_chain)
        rdmol_chain = Chem.AddHs(rdmol_chain)
        for atom in rdmol_chain.GetAtoms():
            info = atom.GetPDBResidueInfo()
            if info is None:
                bonded_atom = atom.GetNeighbors()[0]
                info = bonded_atom.GetPDBResidueInfo()
                atom.SetMonomerInfo(info)
        sys.chain_rdkit[x] = rdmol_chain

    return sys

polymer = terminal_fix(sys)
polymer



In [ ]:
# Randomly assign coordinates to the polymer chains needed for polyply gen_coords
def generate_random_coordinates(mol):
    """
    Assign random 3D coordinates to all atoms in the molecule.
    """
    num_atoms = mol.GetNumAtoms()
    conf = Chem.Conformer(num_atoms)
    for i in range(num_atoms):
        # Generate random x, y, z coordinates in a reasonable range
        x, y, z = random.uniform(-10, 10), random.uniform(-10, 10), random.uniform(-10, 10)
        conf.SetAtomPosition(i, (x, y, z))
    mol.RemoveAllConformers()  # Clear existing conformers
    mol.AddConformer(conf, assignId=True)

# Assign random coordinates
for mol in polymer.chain_rdkit:
    generate_random_coordinates(mol)

# Turns the rdkit molecules into openff molecules for use with the openff toolkit
polymer.chains = [Molecule.from_rdkit(m) for m in polymer.chain_rdkit] # Includes residual monomer and oligomer

# Generate unique atom names for each molecule in the system
for molecule in polymer.chains:
    molecule.generate_unique_atom_names()


In [ ]:
# Parameterize with OpenFF

FFpath = "bespoke_ff.offxml"

topology = Topology.from_molecules(polymer.chains)
interchange = Interchange.from_smirnoff(topology = topology, 
                                        force_field=ForceField(FFpath)
                                        )

In [ ]:
# Checks the total charge of the first chain and shows the parameters used for each atom in the first chain

charge_dict = interchange["Electrostatics"].charges
keys = sorted(charge_dict, key=lambda k: k.atom_indices[0])

# Total charge of one chain (should be 0 for a neutral system)
molecules = list(topology.molecules)
n_atoms = molecules[0].n_atoms
charges = [charge_dict[k].m for k in keys]
total_charge = sum(charges[:n_atoms])
print(f"First chain total charge = {total_charge:.6f} e")


charge_by_atom = {
    key.atom_indices[0]: charge.m
    for key, charge in charge_dict.items()}

# LibraryCharge matches
ff = ForceField(FFpath)
library_handler = ff["LibraryCharges"]
matches = library_handler.find_matches(topology)

print(f"{'AtomID':>6} {'Element':>4} {'Charge':>10} {'Library Charge'}")

all_atoms = list(topology.atoms)

# Loop over atoms in the topology
for atom_index in range(n_atoms):
    
    atom = all_atoms[atom_index]
    charge = charge_by_atom.get(atom_index, None)
    charge_id = "NOT_FOUND"

    # Find the LibraryCharge parameter applied to this atom
    for match, parameter in matches.items():
        if atom_index in match:
            charge_id = parameter.parameter_type.id
            break
    print(
        f"{atom_index:6d} "
        f"{atom.symbol:>4s} "
        f"{charge:10.4f} "
        f"{charge_id}"
    )

In [ ]:
# export to .top file for polyply
interchange.to_top('polymer.top')

# Generate coordinates with polyply (overides the random coordinates)
subprocess.run([
    'polyply', 'gen_coords',
    '-p', 'polymer.top',
    '-name', 'test',
    '-dens', '970',
    '-o', 'polymer.xyz'], 
    check=True
)

In [ ]:
# Read the generated coordinates and set them in the topology

polyply_top_path = 'polymer.xyz'

with open(polyply_top_path, 'r') as file:
    pma_itp_content = file.readlines()

coordinates = []


for i in pma_itp_content[2:-1]:
    parts = i.split()
    #Find atom - make dict
    # Positions (nm → Å)
    x = float(parts[3].strip())*10 
    y = float(parts[4].strip())*10
    z = float(parts[5].strip())*10
    coordinates.append([x,y,z])


coords_arr = np.array(coordinates) 
topology.set_positions(coords_arr * unit.angstrom)


# Box vectors (last line in .xyz file)
box_line = pma_itp_content[-1].split()
box = [float(x) * 10 for x in box_line[:3]]
topology.box_vectors = np.diag(box) * unit.angstrom
print(f"Box vectors set to: {topology.box_vectors}")

# visualise
topology.visualize()

# Create an interchange object
force_field=ForceField(FFpath)
interchange = force_field.create_interchange(topology)
interchange.visualize()



### LAMMPS

This section covers the **energy minimisation**, **heating**, **cooling**, and **production molecular dynamics** stages of the simulation workflow.
LAMMPS is the molecular dynamics engine used here, although it can be replaced with alternatives such as **GROMACS** or **OpenMM** if desired.

#### Energy Minimisation
The initial configuration generated by Polyply is first energy minimised. This reduces large forces and stabilises the system before molecular dynamics begins.

#### Heating and Equilibration
To ensure the polymer chains are able to relax and sample a wider range of conformations, the system is heated above the glass transition temperature of PDMS.

The following protocol is used under the **NPT ensemble**:

1. **0.1 ns heating**
   - Temperature ramped from **300 K → 600 K**
   - Pressure maintained at **1 atm**

2. **0.9 ns equilibration**
   - Temperature held at **600 K**
   - Pressure maintained at **1 atm**

#### Cooling and Equilibration
Following high-temperature equilibration, the system is cooled back to the target simulation temperature.

1. **1 ns cooling**
   - Temperature ramped from **600 K → 300 K**
   - Pressure maintained at **1 atm**

2. **1 ns equilibration**
   - Temperature held at **300 K**
   - Pressure maintained at **1 atm**

This stage allows the polymer density and structure to relax to equilibrium conditions prior to property calculations.

#### Production NPT Simulation

After equilibration, a production simulation is performed in the **NPT ensemble**:

- Temperature: **300 K**
- Pressure: **1 atm**
- Timestep: **0.5 fs**
- Simulation length: **20,000,000 steps**
- Total simulation time: **10 ns**

During production:

- Thermodynamic properties are written every **1000 timesteps**
- Trajectory coordinates are written every **5000 timesteps**
- Volume and enthalpy fluctuations are written every **100 timesteps**

In [ ]:
# Set up LAMMPS input and data files for energy minimisation

def setup_lammps_files_min(
    interchange: Interchange,
    MD_output_file: str,
    data_filename: str,
    input_filename: str,
    logfile: str,
    elements: list,
    n_steps: int = 2000000,
    timestep: float = 0.5,   # fs
    press: float = 1.0,      # atm
    temp: float = 300,
    xyz_frequency: int = 2000,
    thermo_frequency: int = 2000,
):

    data_file = data_filename
    input_file = input_filename

    with open(input_file, "w") as f:

        f.write(f"""
units real
atom_style full
boundary p p p
                
log {logfile}

pair_style lj/cut/coul/long 11 11
pair_modify mix arithmetic tail yes

bond_style harmonic
angle_style harmonic
dihedral_style fourier

read_data {data_file}

thermo_style custom step pe ebond eangle edihed etotal density press temp vol

kspace_style pppm 1e-4

neighbor 2.0 bin
neigh_modify delay 0 every 1 check yes

dump traj all custom 100 {MD_output_file} id element xu yu zu
dump_modify traj sort id element {elements}

thermo {thermo_frequency}

fix relaxbox all box/relax iso 1.0 vmax 0.001

min_style cg
min_modify dmax 0.1

minimize 1.0e-6 1.0e-8 10000 100000

unfix relaxbox
write_data minimized.data
""")

In [ ]:
data_file = f"polymer.data"
interchange.to_lammps_datafile(f"{data_file}")

input_filename = f"lammps_openFF_Min.in"
MD_out_file = f"openFF_min.lammpstrj"
logfile = f"openFF_min.thermo"

reference_masses = {
            "O": 15.99943,
            "Si": 28.08553,
            "C": 12.01078,
            "H": 1.007947,
        }

with open(f"{data_file}") as f:
    lines = f.readlines()

start = lines.index("Masses\n")
masses = []
for line in lines[start+2:]:
    if line.strip() == "":
        break
    print(line.strip())
    masses.append(float(line.strip()[2:]))

elements = []

for mass in masses:
    closest = min(
        reference_masses,
        key=lambda el: abs(reference_masses[el] - mass)
    )
    elements.append(closest)

elements = " ".join(elements)
print(elements)

setup_lammps_files_min(interchange=interchange, 
                    MD_output_file=MD_out_file, 
                    data_filename=data_file, 
                    input_filename=input_filename,
                    logfile=logfile,
                    elements=elements, 
                    temp=300) 


In [ ]:
# Run the minimization in LAMMPS
!lmp -in lammps_openFF_Min.in

In [ ]:
def setup_lammps_files_heat(
    interchange: Interchange,
    MD_output_file: str,
    input_filename: str,
    logfile: str,
    elements: list,
    n_steps: int = 2000000,
    timestep: float = 0.5,   # fs
    press: float = 1.0,      # atm
    temp: float = 300,
    xyz_frequency: int = 2000,
    thermo_frequency: int = 2000,
):

    input_file = input_filename

    with open(input_file, "w") as f:

        f.write(f"""
units real
atom_style full
boundary p p p
                
log {logfile}

# Non-bonded interactions
pair_style lj/cut/coul/long 11 11
pair_modify mix arithmetic tail yes # 'tail yes' adds long-range corrections to energy and pressure for truncated LJ interactions

# Bonded interactions
bond_style harmonic
angle_style harmonic
dihedral_style fourier

read_data minimized.data

thermo_style custom step pe ebond eangle edihed etotal density press temp vol

# long-range electrostatics
kspace_style pppm 1e-4

neighbor 2.0 bin
neigh_modify delay 0 every 1 check yes

timestep {timestep}

velocity all create {temp} 12345 mom yes rot yes dist gaussian

fix remove_drift all momentum 100 linear 1 1 1

dump traj all custom {xyz_frequency} {MD_output_file} id element xu yu zu
dump_modify traj sort id element {elements}

thermo 1000

fix 1 all npt temp 300 600 100.0 iso 1.0 1.0 1000.0
run 200000 #0.1 ns heating npt to 600 K

unfix 1

fix 1 all npt temp 600 600 100.0 iso 1.0 1.0 1000.0
run 1800000 #0.9 ns npt at 600 K and 1 atm

unfix 1

write_restart stage1.restart # writes as .restart to move to next stage of simulation

""")

# 0.1ns heating NpT 300K -> 600K, then 0.9ns NpT at 600K and 1 atm

In [ ]:
input_filename = f"lammps_openFF_MD_heat.in"
MD_out_file = f"openFF_MD_Heat.lammpstrj"
logfile = f"openFF_MD_Heat.thermo"


setup_lammps_files_heat(interchange, 
                    MD_output_file=MD_out_file,  
                    input_filename=input_filename,
                    logfile=logfile,
                    elements=elements, 
                    temp=300) 

In [ ]:
# Run the heating simulation in LAMMPS
!lmp -in lammps_openFF_MD_heat.in

In [ ]:
### Use multiple processors to run faster

# !mpirun -n 4 lmp -in lammps_openFF_MD_heat.in

In [ ]:
def setup_lammps_files_cool (
    interchange: Interchange,
    MD_output_file: str,
    input_filename: str,
    logfile: str,
    elements: list,
    n_steps: int = 2000000,
    timestep: float = 0.5,
    press: float = 1.0,      # atm
    temp: float = 300,
    xyz_frequency: int = 2000,
    thermo_frequency: int = 2000,
):

    input_file = input_filename

    with open(input_file, "w") as f:

        f.write(f"""
units real
atom_style full
boundary p p p
                
log {logfile}

# Non-bonded interactions
pair_style lj/cut/coul/long 11 11
pair_modify mix arithmetic tail yes # 'tail yes' adds long-range corrections to energy and pressure for truncated LJ interactions

# Bonded interactions
bond_style harmonic
angle_style harmonic
dihedral_style fourier

read_restart stage1.restart

thermo_style custom step pe ebond eangle edihed etotal density press temp vol

# long-range electrostatics
kspace_style pppm 1e-4

neighbor 2.0 bin
neigh_modify delay 0 every 1 check yes

timestep {timestep}

fix remove_drift all momentum 100 linear 1 1 1

dump traj all custom {xyz_frequency} {MD_output_file} id element xu yu zu
dump_modify traj sort id element {elements}

thermo 1000

fix 1 all npt temp 600.0 300.0 100.0 iso 1.0 1.0 1000.0
run 2000000 # 1ns cooling npt from 600 K to 300 K

unfix 1

fix 1 all npt temp 300.0 300.0 100.0 iso 1.0 1.0 1000.0

# Collect volume statistics for calculating compressibility
variable V equal vol
variable V2 equal vol*vol

fix volstats all ave/time 1 1000 1000 v_V v_V2 file volume_stats.dat

run 2000000 # 1ns npt at 300 K and 1 atm

unfix 1

write_data cool_eq.data


""")
        
# 1ns cooling NpT from 600K -> 300K, then 1ns NpT at 300K and 1 atm

In [ ]:
input_filename = f"lammps_openFF_MD_cool.in"
MD_out_file = f"openFF_MD_Cool.lammpstrj"
logfile = f"openFF_MD_Cool.thermo"


setup_lammps_files_cool(interchange, 
                    MD_output_file=MD_out_file,  
                    input_filename=input_filename,
                    logfile=logfile,
                    elements=elements, 
                    temp=300)

In [ ]:
# Run the cooling simulation in LAMMPS
!lmp -in lammps_openFF_MD_cool.in

In [ ]:
def setup_lammps_files_prod(
    MD_output_file: str,
    input_filename: str,
    logfile: str,
    elements: list,
    timestep: float = 0.5,
    temp: float = 300.0,
    press: float = 1.0,
    xyz_frequency: int = 5000,
    thermo_frequency: int = 1000,
):
    n_steps = 20000000

    with open(input_filename, "w") as f:
        f.write(f"""
units real
atom_style full
boundary p p p

log {logfile}

# Force field styles
pair_style lj/cut/coul/long 11 11
pair_modify mix arithmetic tail yes

bond_style harmonic
angle_style harmonic
dihedral_style fourier

read_data cool_eq.data

# Long-range electrostatics
kspace_style pppm 1e-4

neighbor 2.0 bin
neigh_modify delay 0 every 1 check yes

timestep {timestep}

thermo_style custom step pe etotal density press temp vol enthalpy
thermo {thermo_frequency}

fix remove_drift all momentum 100 linear 1 1 1

dump traj all custom {xyz_frequency} {MD_output_file} id element xu yu zu
dump_modify traj sort id element {elements}

fix 1 all npt temp 300 300 100.0 iso {press} {press} 1000.0

# Collect volume statistics for calculating compressibility and Heat capacity
variable V equal vol
variable V2 equal vol*vol
variable H equal enthalpy
variable H2 equal enthalpy*enthalpy

fix volstats all ave/time 1 1 100 v_V v_V2 v_H v_H2 file stats.dat
 
run {n_steps}

unfix 1

write_data production_final.data
write_restart production_final.restart
""")

In [ ]:
input_filename = f"lammps_openFF_MD_prod.in"
MD_out_file = f"openFF_MD_prod.lammpstrj"
logfile = f"openFF_MD_prod.thermo"


setup_lammps_files_prod(MD_output_file=MD_out_file,  
                    input_filename=input_filename,
                    logfile=logfile,
                    elements=elements, 
                    temp=300)

## Testing 

Different production runs need to be used for calculating different properties

Density, Heat Capacity, Compressability are all done in Equlibrium MD with NPT over 10 ns

Viscosity is done in Equlibrium MD NVT for 10 ns

In [ ]:
def setup_lammps_files_prod_viscosity(
    MD_output_file: str,
    input_filename: str,
    logfile: str,
    elements: list,
    timestep: float = 0.5,
    temp: float = 300.0,
    press: float = 1.0,
    xyz_frequency: int = 5000,
    thermo_frequency: int = 1000,
):
    n_steps = 20000000

    with open(input_filename, "w") as f:
        f.write(f"""
units real
atom_style full
boundary p p p

log {logfile}

# Force field styles
pair_style lj/cut/coul/long 11 11
pair_modify mix arithmetic tail yes

bond_style harmonic
angle_style harmonic
dihedral_style fourier

read_restart production_final.restart

# Long-range electrostatics
kspace_style pppm 1e-4

neighbor 2.0 bin
neigh_modify delay 0 every 1 check yes

timestep {timestep}

thermo_style custom step pe ebond eangle edihed etotal density press temp vol
thermo {thermo_frequency}

fix remove_drift all momentum 100 linear 1 1 1

dump traj all custom {xyz_frequency} {MD_output_file} id element xu yu zu
dump_modify traj sort id element {elements}

fix 1 all nvt temp 300 300 100.0 

variable pxy equal pxy
variable pxz equal pxz
variable pyz equal pyz

fix SS all ave/correlate 20 200 4000 v_pxy v_pxz v_pyz type auto file shear.acf

run {n_steps}

unfix 1

write_restart viscosity.data
""")

In [ ]:
input_filename = f"lammps_openFF_MD_prod_viscosity.in"
MD_out_file = f"openFF_MD_Prod_viscosity.lammpstrj"
logfile = f"openFF_MD_Prod_viscosity.thermo"


setup_lammps_files_prod(MD_output_file=MD_out_file,  
                    input_filename=input_filename,
                    logfile=logfile,
                    elements=elements, 
                    temp=300)

# Analysis

This section demonstrates how to analyse the molecular dynamics simulations generated during the workflow and extract key thermodynamic properties of PDMS.

### Output Files
- **`.lammpstrj`**
Trajectory file containing atomic coordinates (XYZ data), written every **2000 timesteps**.
 
- **`.thermo`**
Thermodynamic output containing quantities such as:
    - Step
    - Potential Energy (`pe`)
    - Bond Energy (`ebond`)
    - Angle Energy (`eangle`)
    - Dihedral Energy (`edihed`)
    - Total Energy (`etotal`)
    - Density (`density`)
    - Pressure (`press`)
    - Temperature (`temp`)
    - Volume (`vol`)

- **`stats.dat`** 
Contains **volume** and **enthalpy** statistics used to calculate:
    - Compressibility
    - Specific heat capacity




### Notes on Fluctuation Properties
`stats.dat` contains **instantaneous** values for volume and enthapy recorded every **100 timesteps**. This is to avoid infomation loss  that occurs when averaging prior to analysis. Averaging can supress the true variance of volume leading to underestimation of compressability which is dependant on the fluctuations. 

In [ ]:
# Volume histogram

import matplotlib.pyplot as plt
data = np.loadtxt("stats.dat", comments="#")

V = data[:,1]   # <V> values

plt.hist(V, bins=50)
plt.xlabel("Volume (Å³)")
plt.ylabel("Count")

plt.savefig("volume_histogram.png", dpi=300)
plt.show()

In [ ]:
# Expect a gaussian Distribution of volume fluctuations
plt.hist(data[:,1], bins=50);

In [ ]:
# Calculate skewness of the volume distribution
from scipy.stats import skew
print(skew(data[:,1]))

Skew > 0 right tailed \
Skew < 0 left tailed \
Skew = 0 no tail \ 

Skew = 0.095 neglegable difference system likely in equilibrium 


In [ ]:
### Calculate compressibility from volume statistics

import numpy as np

data = np.loadtxt("stats.dat")
V = data[:, 1]  # Extract the volume column
V2 = data[:, 2]  # Extract the volume squared column

mean_V = np.mean(V)
mean_V2 = np.mean(V2)

varV = mean_V2 - mean_V**2

kB = 1.380649e-23  # Boltzmann constant in J/K
T = 300  # Temperature in K

mean_V *= 1e-30 # Å^3 -> m^3
varV *= 1e-60 # Å^6 -> m^6

compressibility = varV/(kB*T*mean_V)  # Calculate compressibility
print(f"Compressibility: {compressibility} Pa^-1")
print(f"Bulk Modulus: {1/compressibility/1e9} GPa")

In [ ]:
### Calculate Specific Heat Capacity from enthalpy in stats.dat

import numpy as np

data = np.loadtxt("stats.dat")
H = data[:, 3]  # Extract the enthalpy column
H2 = data[:, 4]  # Extract the enthalpy squared column (kcal/mol)^2

mean_H = np.mean(H)   # kcal/mol
mean_H2 = np.mean(H2) # (kcal/mol)^2

varH = mean_H2 - mean_H**2

total_mass = sum(atom.mass for atom in topology.atoms).m # Da = g/mol

kB = 1.987204259e-3  # Boltzmann constant in kcal/(mol*K)
T = 300  # Temperature in K

specific_heat = varH/(kB*T**2*total_mass)  # Calculate specific heat capacity
print(f"Specific Heat Capacity: {specific_heat} kcal/(g*K)")
print(f"Specific Heat Capacity: {specific_heat*4184} J/(g*K)")

In [ ]:
## To view the final unwraped structure use ase and aseMolec then visualise in ovito 
from ase.io import read, write
from aseMolec import anaAtoms as aa

atoms = read('openFF_MD_Prod.lammpstrj', '::100')
aa.wrap_molecs(atoms)

write('test.xyz', atoms)

In [ ]:
from aseMolec import pltProps as pp

thermo = pp.loadtxttag('openFF_MD_Prod.thermo')
pp.simpleplot(thermo, 0, 7)